#  C++ Primer : Three Kinds of Memory
# 1. Where a variable lives
* Every variable in a C++ program lives in one of three places, and the difference matters for how long it lives, how big it may be, and --- later in this course --- whether threads share it

| | declared | lifetime | typical size limit |
|---|---|---|---|
| local (stack) | inside a function | until the function returns | a few MB for the whole stack |
| global (static) | outside any function | the whole run of the program | memory of the machine |
| heap | `new` / `delete` | until you `delete` it | memory of the machine |

## 1-1. Local
```
void f() {
  double a[100];   // created on entry to f, gone when f returns
}
```
* Created and destroyed automatically. Fast
* Not initialised: a local variable you do not assign holds whatever was left there
* Returning the address of a local is a classic bug; the variable is gone before the caller uses it

## 1-2. Global
```
double a[100];     // outside any function
void f() { a[0] = 1; }
```
* Exists for the whole run and is visible to every function in the file
* Initialised to zero if you do not say otherwise
* Its size must be known at compile time

## 1-3. Heap
```
double * a = new double[n];   // n decided at run time
...
delete[] a;
```
* `new T[n]` asks for room for `n` objects of type `T` and gives back the address of the first one. `double *` is the type "address of a `double`"
* `a[i]` works exactly as for an array. This is the usual way to get an array whose size is decided at run time
* `delete[] a` gives the memory back. Forgetting it is a memory leak; the program keeps running but keeps growing
* C++ does not collect garbage. There is nothing watching whether the memory is still in use


In [ ]:
%%writefile mem.cc
#include <stdio.h>
#include <stdlib.h>

double g[4];           /* global */

void show(const char * what, void * p) {
  printf("%-8s at %p\n", what, p);
}

int main(int argc, char ** argv) {
  long n = (argc > 1 ? atol(argv[1]) : 4);

  double l[4];                     /* local  */
  double * h = new double[n];      /* heap   */

  for (long i = 0; i < 4; i++) { l[i] = i; }
  for (long i = 0; i < n; i++) { h[i] = i; }

  show("global", (void *)g);
  show("local",  (void *)l);
  show("heap",   (void *)h);
  printf("g[0] = %f (never assigned)\n", g[0]);

  delete[] h;
  return 0;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -Wall mem.cc -o mem

In [ ]:
%%bash
./mem

* The three addresses are far apart: the three kinds of memory live in different regions of the process

* <font color="green">Exercise: try to allocate a large array in each of the three ways and find out which ones survive. Change the local to `double l[100000000];`, run, and see what happens; put it back, and instead run `./mem 100000000` so that the _heap_ array is the big one. Explain the difference.</font>

* Start from this copy, so the program above stays around to compare with


In [ ]:
%%writefile mem_ex.cc
#include <stdio.h>
#include <stdlib.h>

double g[4];                      /* global */

int main(int argc, char ** argv) {
  long n = (argc > 1 ? atol(argv[1]) : 4);

  double l[4];                    /* local -- try double l[100000000]; */
  double * h = new double[n];     /* heap  -- try running with 100000000 */

  for (long i = 0; i < 4; i++) { l[i] = i; }
  for (long i = 0; i < n; i++) { h[i] = i; }

  printf("global at %p\n", (void *)g);
  printf("local  at %p\n", (void *)l);
  printf("heap   at %p\n", (void *)h);

  delete[] h;
  return 0;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -Wall mem_ex.cc -o mem_ex

In [ ]:
%%bash
./mem_ex